# Waddington Landscape Dynamics — Full Reproducible Pipeline

Single source of truth: `adata.h5ad` (raw counts + `Pseudotime` + `X_umap`,
no `X_scVI` yet). Every stage below saves its output to `artifacts/` before
the next stage runs, so nothing depends on Python objects surviving between
sessions — re-running any cell after a kernel restart just reloads from disk.

**Run top to bottom once.** After that, individual sections can be
re-run independently as long as `artifacts/` already has the earlier files.


## 0. Config

Every hyperparameter we locked in during development, in one place.

In [1]:
import os

DATA_PATH = "adata.h5ad"
ARTIFACT_DIR = "artifacts"
SCVI_MODEL_DIR = os.path.join(ARTIFACT_DIR, "scvi_model")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Column names — confirmed against the real dataset during development.
PSEUDOTIME_KEY = "Pseudotime" 
BATCH_KEY = "replicate_id"

# scVI
# If adata.X is NOT raw counts (e.g. it was overwritten with log-normalized
# data during the UMAP/clustering workflow), set this to the layer name
# that holds raw counts instead. None = use adata.X directly.
COUNTS_LAYER = None
N_LATENT = 10
MAX_EPOCHS = 200

# Pseudotime binning / OT
# Equal-COUNT bins (qcut), not equal-width: this dataset's pseudotime is
# heavily skewed (dense progenitor pool at low t, sparse branch tips at
# high t). Equal-width bins would starve the late bins.
N_BINS = 50
# Cost matrix computed in scVI latent space, normalized by its mean BEFORE
# regularizing. reg=0.1 confirmed via effective-target diagnostic: ~13
# soft-matched descendants per cell on average, no row collapsing to 1.
OT_REG = 0.1
# Late-bin velocity has a noisy long tail (small-delta_t artifact from
# sparse downstream target pools near branch tips) — clip before training.
VELOCITY_CLIP_PCT = 99

# Vector field v_theta(t, u1, u2) -> (du1/dt, du2/dt)
VTHETA_HIDDEN_DIM = 64
VTHETA_EPOCHS = 300
VTHETA_BATCH_SIZE = 512
VTHETA_LR = 1e-3
VTHETA_HUBER_DELTA = 1.0   # robust to residual outliers post-clipping

# Space field f_phi(t, u1, u2) -> scVI latent vector
FPHI_HIDDEN_DIM = 128
FPHI_EPOCHS = 300
FPHI_BATCH_SIZE = 512
FPHI_LR = 1e-3

VAL_FRACTION = 0.15
RANDOM_SEED = 0


## 1. Train scVI and produce the canonical latent dataset

`adata.h5ad` has everything except `X_scVI`. We train scVI here, then
immediately write the latent representation, library size, and batch code
back onto the AnnData and save **two** things:

1. The scVI model **with `save_anndata=True`** — this is the fix for the
   earlier problem where the model checkpoint became useless once the
   in-memory AnnData was lost. Never skip this flag again.
2. A standalone `adata_with_latent.h5ad` — the single canonical file every
   later step in this notebook loads from.


In [ ]:
import scanpy as sc
import scvi
import numpy as np

print(f"Loading {DATA_PATH} ...")
adata = sc.read_h5ad(DATA_PATH)
print(adata)


In [ ]:
# scVI needs RAW counts. Quick check before training anything on the wrong scale.
X = adata.layers[COUNTS_LAYER] if COUNTS_LAYER else adata.X
X_sample = np.asarray(X[:50].todense()) if hasattr(X, "todense") else np.asarray(X[:50])
looks_like_counts = np.allclose(X_sample, np.round(X_sample)) and (X_sample >= 0).all()
print("Data looks like raw integer counts:", looks_like_counts)

if not looks_like_counts:
    print("Available layers:", list(adata.layers.keys()))
    raise SystemExit("Set COUNTS_LAYER to the correct raw-counts layer and rerun this cell.")

assert BATCH_KEY in adata.obs.columns, f"{BATCH_KEY} not found in adata.obs"
assert PSEUDOTIME_KEY in adata.obs.columns, f"{PSEUDOTIME_KEY} not found in adata.obs"


In [ ]:
scvi.model.SCVI.setup_anndata(adata, layer=COUNTS_LAYER, batch_key=BATCH_KEY)
model = scvi.model.SCVI(adata, n_latent=N_LATENT)
model.train(max_epochs=MAX_EPOCHS, accelerator="auto")


In [ ]:
# Write latent rep, library size, and batch code back onto adata
adata.obsm["X_scVI"] = model.get_latent_representation()
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
sc.tl.umap(adata)
adata.obs["lib_size"] = np.asarray(X.sum(axis=1)).flatten()
# scvi-tools encodes batch categories in the same order as the pandas
# categorical's .cat.categories (alphabetical by default at setup time).
# This mapping is what we reuse later to decode simulated cells.
adata.obs["batch_code"] = adata.obs[BATCH_KEY].astype("category").cat.codes

model.save(SCVI_MODEL_DIR, overwrite=True, save_anndata=True)

canonical_path = os.path.join(ARTIFACT_DIR, "adata_with_latent.h5ad")
adata.write_h5ad(canonical_path)
print(f"Saved scVI model (with AnnData) to {SCVI_MODEL_DIR}")
print(f"Saved canonical dataset to {canonical_path}")


## 2. Filter + bin pseudotime

This dataset has ~2,500 cells with NaN `Pseudotime` (cells the original
annotation pipeline couldn't confidently place — no DV domain either, so
they're not a real hidden cell type, just unresolved cells). Filtering
happens **once, here** — every downstream stage loads this same filtered
file, so `v_theta` and `f_phi` are guaranteed to train on identical
populations. (Earlier in development, filtering implicitly via `qcut` in
one place but not another caused the two networks to see different cells.)


In [ ]:
import pandas as pd

adata = sc.read_h5ad(os.path.join(ARTIFACT_DIR, "adata_with_latent.h5ad"))

n_before = adata.n_obs
valid_mask = ~adata.obs[PSEUDOTIME_KEY].isna()
print(f"Dropping {(~valid_mask).sum()} / {n_before} cells with NaN {PSEUDOTIME_KEY}")
adata = adata[valid_mask].copy()

adata.obs["pt_bin"] = pd.qcut(adata.obs[PSEUDOTIME_KEY], q=N_BINS, labels=False)
print(adata.obs["pt_bin"].value_counts().sort_index())

filtered_path = os.path.join(ARTIFACT_DIR, "adata_filtered.h5ad")
adata.write_h5ad(filtered_path)
print(f"Saved {filtered_path}")


## 3. Estimate per-cell velocity via optimal transport

Waddington-OT style: for each pair of adjacent pseudotime bins, compute a
Sinkhorn coupling using cost in **scVI latent space** (biological
similarity), then convert that coupling into a velocity vector in
**UMAP space** as displacement toward the probability-weighted *expected*
future position — not a hard nearest-match. That's what keeps the
estimate smooth instead of producing straight-line jumps between bins.


In [ ]:
import ot

adata = sc.read_h5ad(os.path.join(ARTIFACT_DIR, "adata_filtered.h5ad"))

all_t, all_coords, all_velocity = [], [], []

for k in range(N_BINS - 1):
    mask_k  = adata.obs["pt_bin"] == k
    mask_k1 = adata.obs["pt_bin"] == k + 1

    Z_k, Z_k1 = adata.obsm["X_scVI"][mask_k], adata.obsm["X_scVI"][mask_k1]
    U_k, U_k1 = adata.obsm["X_umap"][mask_k], adata.obsm["X_umap"][mask_k1]
    t_k  = adata.obs[PSEUDOTIME_KEY].values[mask_k]
    t_k1 = adata.obs[PSEUDOTIME_KEY].values[mask_k1]

    M = ot.dist(Z_k, Z_k1, metric="sqeuclidean")
    M_norm = M / M.mean()   # normalize scale BEFORE regularizing -- without
                              # this, reg=0.05 collapsed Sinkhorn to near
                              # one-hot matching

    a = np.ones(len(Z_k)) / len(Z_k)
    b = np.ones(len(Z_k1)) / len(Z_k1)
    coupling = ot.sinkhorn(a, b, M_norm, OT_REG)
    coupling_norm = coupling / coupling.sum(axis=1, keepdims=True)

    expected_future_pos = coupling_norm @ U_k1
    expected_future_t   = coupling_norm @ t_k1

    delta_pos = expected_future_pos - U_k
    delta_t   = expected_future_t - t_k
    velocity  = delta_pos / delta_t[:, None]

    all_t.append(t_k)
    all_coords.append(U_k)
    all_velocity.append(velocity)

    print(f"bin {k:2d}->{k+1:2d}  n={mask_k.sum():4d}  mean |v|={np.linalg.norm(velocity,axis=1).mean():.3f}")

all_t        = np.concatenate(all_t)
all_coords   = np.concatenate(all_coords)
all_velocity = np.concatenate(all_velocity)
assert not np.isnan(all_velocity).any(), "NaNs in velocity -- check for near-zero delta_t"


In [ ]:
# Late-bin velocity has a noisy long tail (small-delta_t artifact from
# sparse downstream target pools near branch tips, not real fast biology).
vel_mag = np.linalg.norm(all_velocity, axis=1)
clip_val = np.percentile(vel_mag, VELOCITY_CLIP_PCT)
scale = np.minimum(1, clip_val / vel_mag)
all_velocity_clipped = all_velocity * scale[:, None]

print(f"Velocity magnitude: max before clip={vel_mag.max():.2f}, after={np.linalg.norm(all_velocity_clipped,axis=1).max():.2f}")

velocity_path = os.path.join(ARTIFACT_DIR, "velocity_training_data.npz")
np.savez(velocity_path, t=all_t, coords=all_coords,
         velocity=all_velocity, velocity_clipped=all_velocity_clipped)
print(f"Saved {velocity_path}")


## 4. Train the vector field — v_theta(t, u1, u2) → (du1/dt, du2/dt)

Inputs are standardized (t and u are on very different raw scales).
Huber loss, not plain MSE — it behaves like MSE near zero but like MAE for
large residuals, so any outliers that survived clipping don't dominate
gradient updates.


In [12]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

class VectorField(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 2)
        )
    def forward(self, t, u1, u2):
        x = torch.stack([t, u1, u2], dim=-1)
        return self.net(x)


In [13]:
data = np.load(os.path.join(ARTIFACT_DIR, "velocity_training_data.npz"))
all_t, all_coords, all_velocity_clipped = data["t"], data["coords"], data["velocity_clipped"]

t_mean, t_std = all_t.mean(), all_t.std()
u_mean, u_std = all_coords.mean(axis=0), all_coords.std(axis=0)
v_mean, v_std = all_velocity_clipped.mean(axis=0), all_velocity_clipped.std(axis=0)

t_norm = (all_t - t_mean) / t_std
u_norm = (all_coords - u_mean) / u_std
v_norm = (all_velocity_clipped - v_mean) / v_std

X = np.column_stack([t_norm, u_norm])
Y = v_norm

X_train, X_val, Y_train, Y_val, t_train_raw, t_val_raw = train_test_split(
    X, Y, all_t, test_size=VAL_FRACTION, random_state=RANDOM_SEED
)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
Y_val_t   = torch.tensor(Y_val,   dtype=torch.float32)


In [ ]:
v_theta = VectorField(hidden_dim=VTHETA_HIDDEN_DIM)
optimizer = torch.optim.Adam(v_theta.parameters(), lr=VTHETA_LR)
loss_fn = nn.HuberLoss(delta=VTHETA_HUBER_DELTA)

best_val = float("inf")
for epoch in range(VTHETA_EPOCHS):
    perm = torch.randperm(X_train_t.shape[0])
    total_loss = 0
    for i in range(0, len(perm), VTHETA_BATCH_SIZE):
        idx = perm[i:i+VTHETA_BATCH_SIZE]
        xb, yb = X_train_t[idx], Y_train_t[idx]
        pred = v_theta(xb[:,0], xb[:,1], xb[:,2])
        loss = loss_fn(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * len(idx)

    if epoch % 30 == 0 or epoch == VTHETA_EPOCHS - 1:
        v_theta.eval()
        with torch.no_grad():
            val_pred = v_theta(X_val_t[:,0], X_val_t[:,1], X_val_t[:,2])
            val_loss = loss_fn(val_pred, Y_val_t).item()
        v_theta.train()
        if val_loss < best_val:
            best_val = val_loss
            torch.save(v_theta.state_dict(), os.path.join(ARTIFACT_DIR, "v_theta_best.pt"))
        print(f"epoch {epoch:3d} | train {total_loss/len(perm):.4f} | val {val_loss:.4f}")

np.savez(os.path.join(ARTIFACT_DIR, "norm_stats_vtheta.npz"),
         t_mean=t_mean, t_std=t_std, u_mean=u_mean, u_std=u_std,
         v_mean=v_mean, v_std=v_std)


In [ ]:
# Residual-by-pseudotime-region check -- expect the late-bin region to be
# noticeably worse, a known consequence of the boundary OT artifact above.
v_theta.eval()
with torch.no_grad():
    val_pred = v_theta(X_val_t[:,0], X_val_t[:,1], X_val_t[:,2]).numpy()
per_sample_err = np.linalg.norm(val_pred - Y_val, axis=1)

for lo, hi in zip([0,20,40,60,80], [20,40,60,80,95]):
    mask = (t_val_raw >= lo) & (t_val_raw < hi)
    if mask.sum() > 0:
        print(f"t in [{lo},{hi}): n={mask.sum():4d}  mean residual={per_sample_err[mask].mean():.4f}")


## 5. Train the space field — f_phi(t, u1, u2) → scVI latent vector

Uses exact (cell, latent) pairs — no OT noise involved, so plain MSE is
appropriate here (Huber was specifically for v_theta's noisy finite-
difference targets). **Important:** normalization stats are recomputed
fresh on the full filtered dataset, not reused from v_theta's stats — those
were computed on the OT-restricted subset (bin 49 was never a velocity
*source*, only ever a target, so it's absent from v_theta's training data
but present here).


In [16]:
class SpaceField(nn.Module):
    def __init__(self, latent_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
    def forward(self, t, u1, u2):
        x = torch.stack([t, u1, u2], dim=-1)
        return self.net(x)


In [ ]:
adata = sc.read_h5ad(os.path.join(ARTIFACT_DIR, "adata_filtered.h5ad"))

t_all = adata.obs[PSEUDOTIME_KEY].values
u_all = adata.obsm["X_umap"]
z_all = adata.obsm["X_scVI"]
latent_dim = z_all.shape[1]
print("latent_dim:", latent_dim)

t_mean_f, t_std_f = t_all.mean(), t_all.std()
u_mean_f, u_std_f = u_all.mean(axis=0), u_all.std(axis=0)
# z is already close to mean-0/std-1 per dim by construction -- verify, don't assume
print("z mean/std (first 5 dims):", z_all.mean(axis=0)[:5], z_all.std(axis=0)[:5])

t_norm_f = (t_all - t_mean_f) / t_std_f
u_norm_f = (u_all - u_mean_f) / u_std_f

X_f = np.column_stack([t_norm_f, u_norm_f])
Y_f = z_all

X_train_f, X_val_f, Y_train_f, Y_val_f, t_train_raw_f, t_val_raw_f = train_test_split(
    X_f, Y_f, t_all, test_size=VAL_FRACTION, random_state=RANDOM_SEED
)
X_train_ft = torch.tensor(X_train_f, dtype=torch.float32)
Y_train_ft = torch.tensor(Y_train_f, dtype=torch.float32)
X_val_ft   = torch.tensor(X_val_f,   dtype=torch.float32)
Y_val_ft   = torch.tensor(Y_val_f,   dtype=torch.float32)


In [ ]:
f_phi = SpaceField(latent_dim=latent_dim, hidden_dim=FPHI_HIDDEN_DIM)
optimizer_f = torch.optim.Adam(f_phi.parameters(), lr=FPHI_LR)
loss_fn_f = nn.MSELoss()

best_val_f = float("inf")
for epoch in range(FPHI_EPOCHS):
    perm = torch.randperm(X_train_ft.shape[0])
    total_loss = 0
    for i in range(0, len(perm), FPHI_BATCH_SIZE):
        idx = perm[i:i+FPHI_BATCH_SIZE]
        xb, yb = X_train_ft[idx], Y_train_ft[idx]
        pred = f_phi(xb[:,0], xb[:,1], xb[:,2])
        loss = loss_fn_f(pred, yb)
        optimizer_f.zero_grad(); loss.backward(); optimizer_f.step()
        total_loss += loss.item() * len(idx)

    if epoch % 30 == 0 or epoch == FPHI_EPOCHS - 1:
        f_phi.eval()
        with torch.no_grad():
            val_pred = f_phi(X_val_ft[:,0], X_val_ft[:,1], X_val_ft[:,2])
            val_loss = loss_fn_f(val_pred, Y_val_ft).item()
        f_phi.train()
        if val_loss < best_val_f:
            best_val_f = val_loss
            torch.save(f_phi.state_dict(), os.path.join(ARTIFACT_DIR, "f_phi_best.pt"))
        print(f"epoch {epoch:3d} | train {total_loss/len(perm):.4f} | val {val_loss:.4f}")

np.savez(os.path.join(ARTIFACT_DIR, "norm_stats_fphi.npz"),
         t_mean=t_mean_f, t_std=t_std_f, u_mean=u_mean_f, u_std=u_std_f,
         latent_dim=latent_dim)


In [ ]:
f_phi.eval()
with torch.no_grad():
    val_pred_f = f_phi(X_val_ft[:,0], X_val_ft[:,1], X_val_ft[:,2]).numpy()
per_sample_err_f = np.linalg.norm(val_pred_f - Y_val_f, axis=1)

for lo, hi in zip([0,20,40,60,80], [20,40,60,80,95]):
    mask = (t_val_raw_f >= lo) & (t_val_raw_f < hi)
    if mask.sum() > 0:
        print(f"t in [{lo},{hi}): n={mask.sum():4d}  mean residual={per_sample_err_f[mask].mean():.4f}")


## 6. Simulate a trajectory and decode it to gene expression

Chains everything: integrate `v_theta` forward (Euler-Maruyama) to get a
path, decode each path point through `f_phi` to a latent vector, then
through the **frozen scVI decoder** to expression.

scVI's decoder needs `library_size` and `batch_index` in addition to `z`
(this dataset was trained with `batch_key="replicate_id"`) — simulated
points have no real library/batch, so we pick a reference batch and the
dataset's median library size as defaults (overridable per call).

**Validate the decode wiring on real cells before trusting it on simulated
ones** — this catches API/units mismatches (e.g. scvi-tools version
differences in the `px` distribution's mean attribute) before they
silently produce wrong values inside a trajectory.


In [ ]:
import scvi

loaded_scvi = scvi.model.SCVI.load(SCVI_MODEL_DIR)   # bundled AnnData, self-contained
loaded_scvi.module.eval()
print(scvi.__version__)

adata_ref = loaded_scvi.adata
ref_batch_name = adata_ref.obs[BATCH_KEY].value_counts().idxmax()
ref_batch_code = list(adata_ref.obs[BATCH_KEY].astype("category").cat.categories).index(ref_batch_name)
median_lib_size = np.median(adata_ref.obs["lib_size"].values)
print("Reference batch:", ref_batch_name, "-> code", ref_batch_code)
print("Median library size:", median_lib_size)


In [21]:
def decode_latent_to_expression(z_path, batch_code=ref_batch_code, library_size=None):
    n = z_path.shape[0]
    lib = library_size if library_size is not None else median_lib_size
    z_t = torch.tensor(z_path, dtype=torch.float32)
    library_log = torch.full((n, 1), np.log(lib), dtype=torch.float32)
    batch_idx = torch.full((n, 1), batch_code, dtype=torch.long)
    with torch.no_grad():
        gen_out = loaded_scvi.module.generative(z=z_t, library=library_log, batch_index=batch_idx)
    px = gen_out["px"]
    return px.mu.detach().cpu().numpy() if hasattr(px, "mu") else px.mean.detach().cpu().numpy()


In [ ]:
# --- Validate decode wiring on 5 REAL cells before trusting simulated ones ---
real_idx = np.arange(5)
z_real = adata_ref.obsm["X_scVI"][real_idx]
true_batch_codes = adata_ref.obs[BATCH_KEY].astype("category").cat.codes.values[real_idx]
true_lib = adata_ref.obs["lib_size"].values[real_idx]

decoded = np.array([
    decode_latent_to_expression(z_real[i:i+1], batch_code=true_batch_codes[i], library_size=true_lib[i])[0]
    for i in range(5)
])
actual = np.asarray(adata_ref.X[real_idx].todense()) if hasattr(adata_ref.X, "todense") else adata_ref.X[real_idx]

for i in range(5):
    corr = np.corrcoef(decoded[i], actual[i])[0, 1]
    print(f"cell {i}: correlation decoded vs actual = {corr:.3f}")
print("If these are not solidly positive, stop and debug decode wiring before simulating.")


In [23]:
# --- Reload v_theta / f_phi and define the integrator ---
v_stats = np.load(os.path.join(ARTIFACT_DIR, "norm_stats_vtheta.npz"))
f_stats = np.load(os.path.join(ARTIFACT_DIR, "norm_stats_fphi.npz"))

v_theta = VectorField(hidden_dim=VTHETA_HIDDEN_DIM)
v_theta.load_state_dict(torch.load(os.path.join(ARTIFACT_DIR, "v_theta_best.pt")))
v_theta.eval()

f_phi = SpaceField(latent_dim=int(f_stats["latent_dim"]), hidden_dim=FPHI_HIDDEN_DIM)
f_phi.load_state_dict(torch.load(os.path.join(ARTIFACT_DIR, "f_phi_best.pt")))
f_phi.eval()

adata_filtered = sc.read_h5ad(os.path.join(ARTIFACT_DIR, "adata_filtered.h5ad"))
t_min_obs, t_max_obs = adata_filtered.obs[PSEUDOTIME_KEY].min(), adata_filtered.obs[PSEUDOTIME_KEY].max()
u_min_obs, u_max_obs = adata_filtered.obsm["X_umap"].min(axis=0), adata_filtered.obsm["X_umap"].max(axis=0)


def query_velocity(t, u1, u2):
    t_n  = (t  - v_stats["t_mean"]) / v_stats["t_std"]
    u1_n = (u1 - v_stats["u_mean"][0]) / v_stats["u_std"][0]
    u2_n = (u2 - v_stats["u_mean"][1]) / v_stats["u_std"][1]
    x = torch.tensor([[t_n, u1_n, u2_n]], dtype=torch.float32)
    with torch.no_grad():
        v_n = v_theta(x[:,0], x[:,1], x[:,2]).numpy()[0]
    return v_n * v_stats["v_std"] + v_stats["v_mean"]


def simulate_trajectory(t0, u1_0, u2_0, t_end=None, dt=0.5, noise_scale=0.3, seed=0):
    rng = np.random.default_rng(seed)
    t_end = t_max_obs if t_end is None else t_end
    t, u1, u2 = t0, u1_0, u2_0
    path = [(t, u1, u2)]
    left_bounds = False

    while t < t_end:
        v = query_velocity(t, u1, u2)
        du = v * dt + noise_scale * rng.normal(size=2) * np.sqrt(dt)
        u1_new, u2_new = u1 + du[0], u2 + du[1]
        t_new = t + dt  # pseudotime always advances at rate +1, never learned

        if not (u_min_obs[0] <= u1_new <= u_max_obs[0] and u_min_obs[1] <= u2_new <= u_max_obs[1]):
            left_bounds = True
            u1_new = np.clip(u1_new, u_min_obs[0], u_max_obs[0])
            u2_new = np.clip(u2_new, u_min_obs[1], u_max_obs[1])

        t, u1, u2 = t_new, u1_new, u2_new
        path.append((t, u1, u2))

    if left_bounds:
        print("Note: trajectory touched the boundary of observed coordinates and was clipped.")
    return np.array(path)


def decode_path_to_latent(path):
    t_n  = (path[:,0] - f_stats["t_mean"]) / f_stats["t_std"]
    u1_n = (path[:,1] - f_stats["u_mean"][0]) / f_stats["u_std"][0]
    u2_n = (path[:,2] - f_stats["u_mean"][1]) / f_stats["u_std"][1]
    x = torch.tensor(np.column_stack([t_n, u1_n, u2_n]), dtype=torch.float32)
    with torch.no_grad():
        z = f_phi(x[:,0], x[:,1], x[:,2]).numpy()
    return z


def simulate_and_decode(t0, u1_0, u2_0, **kwargs):
    """End-to-end: starting coordinate -> path -> latent -> decoded expression."""
    path = simulate_trajectory(t0, u1_0, u2_0, **kwargs)
    z_path = decode_path_to_latent(path)
    expr_path = decode_latent_to_expression(z_path)
    return path, z_path, expr_path


In [ ]:
# --- Example run: start near the dense progenitor pool at low pseudotime ---
start_mask = adata_filtered.obs[PSEUDOTIME_KEY] < 5
start_u = adata_filtered.obsm["X_umap"][start_mask.values].mean(axis=0)

path, z_path, expr_path = simulate_and_decode(t0=0.0, u1_0=start_u[0], u2_0=start_u[1])
print("path shape:", path.shape, "| latent shape:", z_path.shape, "| expression shape:", expr_path.shape)

np.savez(os.path.join(ARTIFACT_DIR, "example_trajectory.npz"),
         path=path, z_path=z_path, expr_path=expr_path, gene_names=adata_filtered.var_names.values)
print(f"Saved example trajectory to {os.path.join(ARTIFACT_DIR, 'example_trajectory.npz')}")


## 7. Quick look — 3D path + a few marker genes over the trajectory

A lightweight sanity-check plot before building anything more polished.


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

fig = plt.figure(figsize=(14, 5))

ax = fig.add_subplot(1, 2, 1, projection="3d")
bg = adata_filtered.obsm["X_umap"]
bg_t = adata_filtered.obs[PSEUDOTIME_KEY].values
ax.scatter(bg_t, bg[:,0], bg[:,1], s=2, alpha=0.15, c=bg_t, cmap="viridis")
ax.plot(path[:,0], path[:,1], path[:,2], color="red", linewidth=2)
ax.set_xlabel("pseudotime"); ax.set_ylabel("UMAP1"); ax.set_zlabel("UMAP2")
ax.set_title("Simulated path through the landscape")

ax2 = fig.add_subplot(1, 2, 2)
gene_names = adata_filtered.var_names.values
# Swap these for genes relevant to your dataset / fate of interest
genes_to_plot = [g for g in [
    "Sox2",
    "Olig2",
    "Nkx6-1",
    "Pax6"
] if g in gene_names]
for g in genes_to_plot:
    gi = list(gene_names).index(g)
    ax2.plot(path[:,0], expr_path[:, gi], label=g)
ax2.set_xlabel("pseudotime"); ax2.set_ylabel("decoded expression")
ax2.legend()
ax2.set_title("Decoded expression along the path")

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "trajectory_preview.png"), dpi=150)
plt.show()
